# This workflow demonstrates the workflow to burn scars from Harmonized Landsat and Sentinel-2 (HLS) imagery from Microsoft Planetary Computer and pre-trained deep learning model from ArcGIS Online

The workflow includes the following steps:
1. Find the HLS landsat scenes from 2020 that covers the Apple and El Dorado Fires in southern California
2. Create a RasterCollection from all the scenes
3. Mosaic all scenes from the RasterCollection together to create the input raster for mapping
4. Use the Classify Pixels Using DeepLearning tool to map the burn scars and save output

##### Note: It's highly recommended to use a NVIDIA GPU-enabled machine to perform this workflow. You may need to install the driver manually (https://learn.microsoft.com/en-us/azure/virtual-machines/windows/n-series-driver-setup)
##### The output may not appear correct if you use GPU for processing but have not installed the driver

#### Estimated running time: 1 hr (with GPU)

### Import relevant modules and log in to ArcGIS Online

In [ ]:
import arcpy
from arcpy import AIO
import getpass
from arcgis.gis import GIS

In [ ]:
# Log in to ArcGIS online to access deep learning models
portal="https://arcgis.com"
username = getpass.getpass('Enter AGOL username:')
password = getpass.getpass('Enter AGOL password: ')
params = arcpy.SignInToPortal(portal,username,password)
gis = GIS(portal)

### Create a RasterCollection of potential scenes from planetary computer archive

In [ ]:
a = AIO(r'C:\AMPC_Resources\ACS_Files\esrims_pc_hls2-l30.acs')

In [ ]:
query = {
    "collections": ["hls2-l30"], 
    "bbox": [-117.133257, 34.019967, -116.765293, 34.159832], 
    "datetime": "2020-11-01/2020-11-30", 
    "limit": 1000, 
          }

In [ ]:
rc = arcpy.ia.RasterCollection.fromSTACAPI(stac_api="https://planetarycomputer.microsoft.com/api/stac/v1",
                                           query=query,
                                           attribute_dict={
                                               "Name":"id",
                                               "Cloud Cover":"eo:cloud_cover",
                                               "StdTime":"datetime",
                                               "Platform":"platform",
                                               "Spatial Reference":"proj:epsg",
                                               "Extent": "bbox",
                                           },
                                           context = {"assetManagement": ["B02", "B03", "B04", "B05", "B06", "B07"]}
                                          )

In [ ]:
rc

### Create a input mosaic and map burn scars using a deep learning model from ArcGIS Online

In [ ]:
input_raster = rc.mosaic()

Now we need to use ArcGIS API for Python to search for a suitable deep learning model (dlpk) on ArcGIS Online
<br>
Documentation on how to perform a search on ArcGIS Online: https://developers.arcgis.com/python/guide/accessing-and-creating-content/#searching-for-content-in-arcgis-online

In [ ]:
# Search for a suitable deep learning model on ArcGIS Online
query = 'Prithvi - Burn Scars Segmentation dlpk, owner:esri_analytics'
classification_models = gis.content.search(query = query, max_items = 30, outside_org = True)
for model in classification_models:
    display(model)

Here we want to use Prithvi - Burn Scars Segmentation model.
<br>
The model link is https://www.arcgis.com/home/item.html?id=9af7af28dd91473bbc8ad40942e74563
<br>
We need to obtain the model id which is 9af7af28dd91473bbc8ad40942e74563

In [ ]:
deeplearning_model_id = '9af7af28dd91473bbc8ad40942e74563'

In [ ]:
# Classify the image using a deep learning model
# Make sure you have installed the GPU driver if you specify processorType="GPU",  otherwise specify processorType="CPU"
with arcpy.EnvManager(processorType="GPU"): 
    out_classified_raster = arcpy.ia.ClassifyPixelsUsingDeepLearning(
        in_raster = input_raster,
        in_model_definition = "https://www.arcgis.com/sharing/rest/content/items/" + deeplearning_model_id,
        arguments = "padding 56;batch_size 4;test_time_augmentation True;predict_background True", 
        processing_mode = "PROCESS_AS_MOSAICKED_IMAGE",
        out_classified_folder = None,
        out_featureclass = None,
        overwrite_attachments = "NO_OVERWRITE"
    )
    # Save output to a local file
    out_classified_raster.save(r"C:\temp\burnscar.tif")

In [ ]:
# Display the output
out_classified_raster